# CAC 40 TR — 2× Leveraged Margin Account + IVol EMA Exit

**Strategy:** hold CAC 40 Total Return 2× on margin (IBKR EUR), financing with €STR + 1.50 % spread.  
**Exit signal:** go flat when IVol rises above its EMA(N); re-enter when IVol falls back below.  
**Signal = EMA(N) − IVol**: positive = vol calming = in market.

| Parameter | Description |
|---|---|
| `EMA_PERIOD` | IVol EMA span (days) |
| `IBKR_SPREAD` | IBKR EUR spread over €STR (default 1.50 %) |
| θ slider | Sensitivity tune — how far IVol must be below EMA before entering |

In [22]:
import sys
import pandas as pd, numpy as np
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

# ── Signum charting ──
sys.path.insert(0, r"C:\Personal\Business & Investments\Python codes")
from signum import Chart, Dashboard

# ── Sfera DB ──
import sfera_db

In [23]:

# ── Load CAC IVol + LVC ETF + CACT total return + €STR overnight rates ──

IBKR_SPREAD = 0.015   # +1.50 % over €STR (IBKR Pro EUR, balance < €90 K tier)

cac_ivol   = sfera_db.index_ivol('CAC')[['ivol']]
cac_tr_raw = sfera_db.read_table('index_total_return', ticker='CACT')[['close_price']]
cac_tr_px  = cac_tr_raw.rename(columns={'close_price': 'close'})

LVC_XLSX = Path(r"C:\Personal\Business & Investments\Trading portfolio\Strategies\ETF TKAN\Data\Archive\LVC_daily.xlsx")
lvc_raw = (
    pd.read_excel(LVC_XLSX, parse_dates=['Date'])
    .rename(columns={'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low',
                     'Close': 'close', 'Volume': 'volume'})
    .set_index('date')
    .sort_index()
)

# Align on common dates: LVC ∩ IVol ∩ CACT
common_lvc = cac_ivol.index.intersection(lvc_raw.index).intersection(cac_tr_px.index)
lvc = pd.DataFrame({
    'close':      lvc_raw.loc[common_lvc, 'close'],
    'ivol':       cac_ivol.loc[common_lvc, 'ivol'],
    'ret':        lvc_raw.loc[common_lvc, 'close'].pct_change(),
    'cac_tr_ret': cac_tr_px.loc[common_lvc, 'close'].pct_change(),
}).dropna()

# ── €STR / EONIA overnight rates ──
RATES_CSV = Path(r"C:\Personal\Business & Investments\Trading portfolio\ForgeFolio\data\eur_overnight_rate.csv")
_rates = (
    pd.read_csv(RATES_CSV, parse_dates=['date'], index_col='date')
    ['rate_pct']
    .reindex(lvc.index, method='ffill')
)
# Daily borrow on 1× equity loan: (€STR/100 + spread) / 252
lvc['borrow_daily'] = (_rates / 100 + IBKR_SPREAD) / 252
# Margin return on equity: 2× CAC TR gain minus financing cost
lvc['margin_ret']   = 2 * lvc['cac_tr_ret'] - lvc['borrow_daily']
lvc = lvc.dropna(subset=['margin_ret'])

def _sharpe(r): return r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0
def _mdd(eq):   return ((eq - eq.cummax()) / eq.cummax()).min() * 100

lvc_bh_eq      = (1 + lvc['ret']).cumprod()
lvc_bh_sharpe  = _sharpe(lvc['ret'])
lvc_bh_mdd     = _mdd(lvc_bh_eq)

margin_bh_eq     = (1 + lvc['margin_ret']).cumprod()
margin_bh_sharpe = _sharpe(lvc['margin_ret'])
margin_bh_mdd    = _mdd(margin_bh_eq)

# Execution: signal at close[T] → MOC fill close[T] → earn close[T+1]/close[T]-1
print(f"Date range  : {lvc.index[0].date()} → {lvc.index[-1].date()}  ({len(lvc)} days)")
print(f"LVC B&H     Sharpe {lvc_bh_sharpe:.3f}  Total {(lvc_bh_eq.iloc[-1]-1)*100:+.1f}%  MaxDD {lvc_bh_mdd:.1f}%")
print(f"Margin B&H  Sharpe {margin_bh_sharpe:.3f}  Total {(margin_bh_eq.iloc[-1]-1)*100:+.1f}%  MaxDD {margin_bh_mdd:.1f}%")
print(f"Borrow rate : €STR + {IBKR_SPREAD*100:.2f}% = {(_rates.iloc[-1]/100 + IBKR_SPREAD)*100:.2f}%/yr today")


Date range  : 2008-06-16 → 2026-03-20  (4549 days)
LVC B&H     Sharpe 0.318  Total +126.1%  MaxDD -75.2%
Margin B&H  Sharpe 0.336  Total +154.1%  MaxDD -75.6%
Borrow rate : €STR + 1.50% = 3.43%/yr today


## IVol Z-score Exit — LVC vs 2× Margin

Long when **IVol Z-score (150d window) < 1.0** — regime-normalised threshold that adapts to secular vol shifts.  
Applied to both **LVC ETF** (2× CAC, Amundi) and the **synthetic 2× CAC TR margin account** (IBKR, €STR + 1.50%).


In [24]:

# ── IVol Z-score 150d signal — applied to LVC ETF and 2× Margin ──
# Long when rolling Z-score of IVol < 1.0 (IVol below regime-normalised threshold).
# Execution: MOC same day — signal at close[T] → earn close[T+1]/close[T]-1.

Z_WINDOW = 150   # best window from ivol_hedge_signal.ipynb
Z_THRESH = 1.0

ivol    = lvc['ivol']
rm      = ivol.rolling(Z_WINDOW).mean()
rs      = ivol.rolling(Z_WINDOW).std()
z_score = (ivol - rm) / rs
pos     = ((z_score < Z_THRESH).fillna(True)).astype(float)
lagged  = pos.shift(1).fillna(0)

# LVC strategy
sr_lvc   = lagged * lvc['ret']
eq_lvc   = (1 + sr_lvc).cumprod()
sh_lvc   = _sharpe(sr_lvc)

# Margin strategy
sr_mgn   = lagged * lvc['margin_ret']
eq_mgn   = (1 + sr_mgn).cumprod()
sh_mgn   = _sharpe(sr_mgn)

pct_in   = lagged.mean() * 100
print(f"Z{Z_WINDOW}d signal — {pct_in:.0f}% in market")
print(f"  LVC    Sharpe {sh_lvc:.3f}  Total {(eq_lvc.iloc[-1]-1)*100:+.1f}%  MaxDD {_mdd(eq_lvc):.1f}%")
print(f"  Margin Sharpe {sh_mgn:.3f}  Total {(eq_mgn.iloc[-1]-1)*100:+.1f}%  MaxDD {_mdd(eq_mgn):.1f}%")
print(f"  (LVC B&H Sharpe {lvc_bh_sharpe:.3f} | Margin B&H Sharpe {margin_bh_sharpe:.3f})")

t      = lvc.index
pos_df = pd.DataFrame({'time': t, 'position': lagged.values})
z_df   = pd.DataFrame({'time': t, 'value': z_score.values})

p_price = (
    Chart(height=280)
    .line(pd.DataFrame({'time': t, 'value': lvc['close'].values}),
          name='LVC 2×', color='#a78bfa', width=2)
    .shade(pos_df, position_col='position', color='#34d399', opacity=0.10)
)
p_z = (
    Chart(height=180)
    .line(z_df, name=f'IVol Z-score {Z_WINDOW}d', color='#fb923c', width=1)
    .price_line(price=Z_THRESH, title=f'Z={Z_THRESH}',
                color='#f87171', line_style=2, line_width=1)
)
p_eq = (
    Chart(height=200)
    .line(pd.DataFrame({'time': t, 'value': eq_lvc.values}),
          name=f'LVC + Z{Z_WINDOW}d  ({sh_lvc:.3f})', color='#34d399', width=2)
    .line(pd.DataFrame({'time': t, 'value': eq_mgn.values}),
          name=f'Margin + Z{Z_WINDOW}d  ({sh_mgn:.3f})', color='#f59e0b', width=2)
    .line(pd.DataFrame({'time': t, 'value': lvc_bh_eq.values}),
          name=f'LVC B&H  ({lvc_bh_sharpe:.3f})', color='#6b7280', width=1)
    .line(pd.DataFrame({'time': t, 'value': margin_bh_eq.values}),
          name=f'Margin B&H  ({margin_bh_sharpe:.3f})', color='#94a3b8', width=1)
)

Dashboard(
    panes=[p_price, p_z, p_eq],
    titles=[
        f'LVC 2×  (green = in market)  |  Z{Z_WINDOW}d: {pct_in:.0f}% in market',
        f'IVol Z-score ({Z_WINDOW}d window) — long when Z < {Z_THRESH}',
        'LVC B&H  vs  LVC+Z150d  vs  Margin B&H  vs  Margin+Z150d',
    ],
    theme='dark'
)


Z150d signal — 81% in market
  LVC    Sharpe 0.423  Total +340.8%  MaxDD -58.7%
  Margin Sharpe 0.427  Total +354.9%  MaxDD -58.4%
  (LVC B&H Sharpe 0.318 | Margin B&H Sharpe 0.336)


## Annual Performance Table

In [25]:

# ── Annual return comparison — LVC & Margin B&H vs IVol Z-score / EMA strategies ──
from IPython.display import display

WINDOWS     = [150, 170, 200]   # Z-score lookback windows to compare
EMA_PERIODS = [8, 9, 10]        # legacy EMA comparison

def _z_pos(w):
    iv = lvc['ivol']; rm = iv.rolling(w).mean(); rs = iv.rolling(w).std()
    return ((iv - rm) / rs < 1.0).fillna(True).astype(float)

def _ema_pos(n):
    iv = lvc['ivol']
    return (iv < iv.ewm(span=n).mean()).astype(float)

def _lag(p): return p.shift(1).fillna(0)

def _mdd_r(s): return _mdd((1 + s).cumprod())

def _ir(s, b):
    exc = s - b
    return exc.mean() / exc.std() * np.sqrt(252) if exc.std() > 0 else np.nan

bh_lvc = lvc['ret']
bh_mgn = lvc['margin_ret']

strats = {'LVC B&H': bh_lvc, 'Mgn B&H': bh_mgn}
for w in WINDOWS:
    p = _lag(_z_pos(w))
    strats[f'LVC Z{w}d'] = p * bh_lvc
    strats[f'Mgn Z{w}d'] = p * bh_mgn
for n in EMA_PERIODS:
    strats[f'LVC EMA{n}'] = _lag(_ema_pos(n)) * bh_lvc

def _ann_ret(s): return (1 + s).resample('YE').prod() - 1

annual_bh = _ann_ret(bh_lvc)
years     = annual_bh.index.year

# ── Annual returns body ──
tbl = pd.DataFrame({
    col: (_ann_ret(sr) * 100).round(1).values
    for col, sr in strats.items()
})
tbl.index = years

# ── Summary rows ──
def _tot(s): return ((1 + s).prod() - 1) * 100

tbl.loc['Total %']        = {c: round(_tot(s),         1) for c, s in strats.items()}
tbl.loc['Sharpe']         = {c: round(_sharpe(s),      2) for c, s in strats.items()}
tbl.loc['MaxDD %']        = {c: round(_mdd_r(s),       1) for c, s in strats.items()}
tbl.loc['IR vs LVC B&H']  = {c: (round(_ir(s, bh_lvc), 2) if c != 'LVC B&H' else np.nan)
                               for c, s in strats.items()}

all_cols = list(strats.keys())
display(
    tbl.style
    .background_gradient(subset=pd.IndexSlice[years, all_cols], cmap='RdYlGn',
                         vmin=-50, vmax=80, axis=None)
    .format(lambda v: f'{v:+.1f}' if isinstance(v, (float, int)) and pd.notna(v) else '',
            subset=pd.IndexSlice[years, all_cols])
    .format(lambda v: f'{v:+.2f}' if isinstance(v, (float, int)) and pd.notna(v) else '',
            subset=pd.IndexSlice[['Total %', 'Sharpe', 'MaxDD %', 'IR vs LVC B&H'], all_cols])
    .set_caption('Annual return % — LVC & Margin B&H vs IVol Z-score / EMA strategies  (1-day lag)')
)


,LVC B&H,Mgn B&H,LVC Z150d,Mgn Z150d,LVC Z170d,Mgn Z170d,LVC Z200d,Mgn Z200d,LVC EMA8,LVC EMA9,LVC EMA10
2008,-58.2,-59.0,+0.0,+0.0,+0.0,+0.0,+0.0,+0.0,-55.3,-52.4,-48.2
2009,+35.5,+48.2,+54.4,+65.7,+56.2,+70.1,+73.1,+89.6,+10.2,+20.4,+24.3
2010,-13.1,-6.2,-25.3,-21.1,-24.8,-20.7,-24.4,-20.3,+6.5,+9.1,+7.6
2011,-33.3,-32.6,-34.6,-35.4,-32.4,-33.2,-20.7,-20.7,-51.4,-54.4,-51.0
2012,+35.4,+36.4,+32.1,+32.9,+35.4,+36.4,+35.4,+36.4,-3.6,+2.8,+2.1
2013,+44.3,+43.2,+29.0,+26.4,+30.0,+28.2,+30.2,+28.2,+7.9,+2.7,-2.9
2014,+0.7,+1.1,+1.8,+0.6,+0.2,-0.7,-1.3,-2.3,+8.1,+1.1,+4.6
2015,+18.9,+17.4,-18.6,-20.1,-24.0,-25.3,-31.6,-32.9,-5.0,-9.2,-10.4
2016,+11.7,+11.9,-5.8,-6.6,-2.6,-3.8,+2.9,+2.1,+3.5,-7.5,-15.4
2017,+24.6,+24.2,+10.8,+10.4,+11.0,+10.4,+10.3,+9.7,+6.4,+4.1,+6.7


In [26]:

# ── IVol Z-score 150d on LVC — OHLC master dataframe + trade log ──
#
# EXECUTION MODEL: MOC (Market-On-Close) same day
#   signal[T] = z_score[T] < 1.0  → fires at close[T]
#   BUY  at close[T]   (MOC auction, same session)
#   SELL at close[S]   (MOC auction on exit signal day)
#   Return: close[T+1]/close[T]-1  (standard c2c while in position)

_Z_WIN = 150

_ohlc    = lvc_raw.loc[lvc.index, ['open', 'high', 'low', 'close']].copy()
_ivol    = lvc['ivol']
_rm      = _ivol.rolling(_Z_WIN).mean()
_rs      = _ivol.rolling(_Z_WIN).std()
_zscore  = (_ivol - _rm) / _rs
_sig     = ((_zscore < 1.0).fillna(True)).astype(int)
_pos     = _sig.shift(1).fillna(0).astype(int)   # in position on day T → bought at close[T-1]

_close_ret = lvc['ret']                           # close[T]/close[T-1]-1
_sr        = _pos.astype(float) * _close_ret      # simple c2c while in position
_eq        = (1 + _sr).cumprod()

_sh_val  = _sr.mean() / _sr.std() * np.sqrt(252)
_tot_val = (_eq.iloc[-1] - 1) * 100
_mdd_val = ((_eq - _eq.cummax()) / _eq.cummax()).min() * 100
_pct_val = _pos.mean() * 100
print(f"IVol Z-score {_Z_WIN}d — MOC exec  |  Sharpe {_sh_val:.3f}  Total {_tot_val:+.1f}%  MaxDD {_mdd_val:.1f}%  {_pct_val:.0f}% in market")
print(f"LVC B&H                          |  Sharpe {lvc_bh_sharpe:.3f}  Total {(lvc_bh_eq.iloc[-1]-1)*100:+.1f}%")

# ── Exec prices: close of signal day ──
_sig_diff = _sig.diff()
_trade    = pd.Series('', index=lvc.index, dtype=str)
_trade[_sig_diff ==  1] = 'BUY'
_trade[_sig_diff == -1] = 'SELL'
_exec_px  = pd.Series(np.nan, index=lvc.index)
_exec_px[_sig_diff != 0] = _ohlc['close'][_sig_diff != 0]   # close[T] — MOC fill price

# ── master_lvc: daily OHLC + signal/trade/position/returns ──
master_lvc = pd.DataFrame({
    'open':        _ohlc['open'].round(3),
    'high':        _ohlc['high'].round(3),
    'low':         _ohlc['low'].round(3),
    'close':       _ohlc['close'].round(3),
    'ivol':        _ivol.round(2),
    'z_150d':      _zscore.round(3),
    'signal':      _sig,
    'trade':       _trade,
    'exec_price':  _exec_px.round(3),   # close[T] — MOC fill price
    'position':    _pos,
    'lvc_ret_%':   (_close_ret * 100).round(3),
    'strat_ret_%': (_sr * 100).round(3),
    'portfolio':   _eq.round(4),
})
master_lvc.index.name = 'date'

# ── trades_lvc: trade-level P&L ──
_buys  = master_lvc[master_lvc['trade'] == 'BUY']
_sells = master_lvc[master_lvc['trade'] == 'SELL']
_trades_list = []
for buy_dt, buy_row in _buys.iterrows():
    fut = _sells[_sells.index > buy_dt]
    if len(fut) == 0:
        sell_dt = master_lvc.index[-1]
        sell_px = master_lvc.loc[sell_dt, 'close']
    else:
        sell_dt = fut.index[0]
        sell_px = fut.iloc[0]['exec_price']
    buy_px = buy_row['exec_price']
    eq_in  = master_lvc.loc[buy_dt,  'portfolio']
    eq_out = master_lvc.loc[sell_dt, 'portfolio']
    _trades_list.append({
        'Signal BUY':    buy_dt.date(),
        'Signal SELL':   sell_dt.date(),
        'Calendar days': (sell_dt - buy_dt).days,
        'Buy close':     round(buy_px,  3),
        'Sell close':    round(sell_px, 3),
        'Price ret %':   round((sell_px / buy_px - 1) * 100, 2),
        'Port. in':      round(eq_in,   4),
        'Port. out':     round(eq_out,  4),
        'Port. ret %':   round((eq_out / eq_in - 1) * 100, 2),
    })

trades_lvc = pd.DataFrame(_trades_list)
print(f"\nmaster_lvc: {len(master_lvc)} rows  |  trades_lvc: {len(trades_lvc)} trades")


IVol Z-score 150d — MOC exec  |  Sharpe 0.423  Total +340.8%  MaxDD -58.7%  81% in market
LVC B&H                          |  Sharpe 0.318  Total +126.1%

master_lvc: 4549 rows  |  trades_lvc: 103 trades
